# Thinking Spatially: Mapping Health Facilities

**Lesson 5: GeoPandas** — from tables to maps.

**Scenario.** You advise South Africa's health department. You have an OpenStreetMap export of
health facilities (hospitals, clinics, pharmacies, ...) and the official province boundaries. The
question: *how are facilities distributed across the nine provinces, and where are the gaps?*

**Data** (all local, under `data/0_raw/south_africa/geo_data/`)
- `south-africa.geojson` — OSM health facilities (points and building polygons)
- `zaf_admin_boundaries.geojson/zaf_admin1.geojson` — province boundaries (COD-AB)

Remember: a `GeoDataFrame` is a normal DataFrame with a `geometry` column, and a spatial join
matches rows by *where they are* — is this point **within** that polygon?

In [ ]:
pip install geopandas contextily mapclassify

In [ ]:
import sys
from pathlib import Path
import pandas as pd

import geopandas as gpd
import contextily as cx
import matplotlib.pyplot as plt

project_root = Path.cwd()
while not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.append(str(project_root))
from src.utilities.project_paths import RAW_DIR

GEO_DIR           = RAW_DIR / 'geo_data'
HEALTH_GEOJSON    = GEO_DIR / 'south-africa.geojson'
PROVINCES_GEOJSON = GEO_DIR / 'zaf_admin_boundaries.geojson' / 'zaf_admin1.geojson'

pd.set_option('display.float_format', '{:,.1f}'.format)
print('geopandas', gpd.__version__, '| contextily', cx.__version__)

---
# Part A — Loading Spatial Data

Two layers: the **facilities** (points) and the **provinces** (polygons). Along the way we hit a
real-world snag — the facilities layer mixes points and polygons — and fix it.

## A1. Load the health facilities

`gpd.read_file()` reads a GeoJSON straight into a `GeoDataFrame`. Inspect the CRS, the columns, and
the **geometry types**.

In [ ]:
# TODO: load HEALTH_GEOJSON with gpd.read_file(...)
health = ...

print('CRS        :', health.crs)
print('rows, cols :', health.shape)
print('geom types :', health.geometry.geom_type.value_counts().to_dict())
display(health[['name', 'amenity', 'healthcare', 'operator']].head())

## A2. Load the province boundaries

The COD-AB file has one row per province. Keep the name (`adm1_name`), the area (`area_sqkm`), and
the geometry.

In [ ]:
# TODO: load PROVINCES_GEOJSON, keeping columns ['adm1_name','adm1_pcode','area_sqkm','geometry']
provinces = ...

print('CRS   :', provinces.crs, '| shape :', provinces.shape)
display(provinces[['adm1_name', 'area_sqkm']])

# TODO: plot the provinces (color='lightgray', edgecolor='white')

## A3. Fix the mixed geometry, and tidy the facility type

Two clean-ups:
1. Collapse every facility to a single point with `.representative_point()` (a point guaranteed to
   lie inside the shape — works for both points and polygons).
2. Fold the messy `amenity` values into a small `facility_type` category.

In [ ]:
# TODO 1: collapse mixed geometry to points
#   health_points = health.copy()
#   health_points['geometry'] = health.geometry.representative_point()
health_points = ...

# TODO 2: fold amenity into a small 'facility_type' category
main_types = ['hospital', 'clinic', 'pharmacy', 'doctors', 'dentist']
#   health_points['facility_type'] = health_points['amenity'].where(
#       health_points['amenity'].isin(main_types), 'other')

print('geom types now:', health_points.geometry.geom_type.value_counts().to_dict())
display(health_points['facility_type'].value_counts().to_frame())

# TODO: plot provinces (lightgray) then the facility points (ax=..., markersize=3)

---
# Part B — Spatial Join: Which Province Is Each Facility In?

The facilities have coordinates but no province label. A spatial join recovers it — *is this point
`within` that polygon?* — the "Spatial VLOOKUP".

## B1. Tag each facility with its province

In [ ]:
# TODO: spatial join health_points INTO provinces (predicate='within', how='left')
tagged = ...

print('unmatched:', tagged['adm1_name'].isna().sum())
display(tagged[['name', 'facility_type', 'adm1_name']].head())

## B2. Count facilities per province and per type

Now it is a plain DataFrame — use `groupby`.

In [ ]:
# TODO:
# 1. by_province = count of rows per adm1_name (groupby().size()), sorted desc
# 2. by_type     = counts per ['adm1_name','facility_type'], then .unstack(fill_value=0)
by_province = ...
by_type = ...
display(by_province)
display(by_type)

---
# Part C — Counts vs Density

Raw counts favour big, busy provinces. Normalising by area tells a different — and often fairer —
story.

## C1. Attach counts to the polygons and compute density

Merge the per-province counts back onto the province polygons, then compute facilities per
10,000 km² using `area_sqkm`.

In [ ]:
# TODO:
# 1. merge by_province onto provinces (on='adm1_name', how='left') -> prov_map
# 2. add prov_map['per_10000km2'] = n_facilities / area_sqkm * 10_000
prov_map = ...

display(prov_map[['adm1_name', 'n_facilities', 'area_sqkm', 'per_10000km2']]
        .sort_values('per_10000km2', ascending=False))

---
# Part D — Multi-Layer Maps

Layer cake: basemap at the bottom, province choropleth in the middle, facility points on top.
Convert everything to **Web Mercator (EPSG:3857)** first so `contextily` basemaps align.

## D1. Facilities map — choropleth + points by type

In [ ]:
prov_web = prov_map.to_crs(3857)
pts_web  = health_points.to_crs(3857)

fig, ax = plt.subplots(figsize=(11, 11))

# TODO: build the layer cake:
# 1. prov_web.plot(column='n_facilities', cmap='YlGnBu', legend=True, zorder=1)
# 2. pts_web.plot(column='facility_type', markersize=6, cmap='Set1', legend=True, zorder=3)
# 3. cx.add_basemap(ax, source=cx.providers.CartoDB.Positron)   # wrap in try/except
# 4. ax.set_axis_off()
plt.show()

## D2. Density map — the same data, normalised

Colour provinces by facilities per 10,000 km² instead of raw counts, and label each province.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 9))

# TODO:
# 1. prov_web.plot(column='per_10000km2', cmap='OrRd', legend=True, zorder=1)
# 2. label each province at its representative_point() with adm1_name + density
# 3. add a basemap (try/except); ax.set_axis_off()
plt.show()